# DA-GPS moderate add-ons training — Google Colab

Clean Colab runner for the **moderate** add-on config (hybrid tap loss, regulator territory attention bias, voltage violation MSE). **Current recommended production config** after ep1 smoke comparison.

**Before you start:** `Runtime > Change runtime type > GPU` (T4 is fine). Mount Google Drive when prompted in the training cell.

## Moderate add-ons (recommended)
Best ep1 performer on val_tot / val_reg / tap_acc. Coefficients: ordinal α=1.75, territory β=7.0, violation V=6.0.

| Flag | Value |
|------|-------|
| `ATTN_REG_TERRITORY_BETA` | 7.0 |
| `REG_ORDINAL_ALPHA` | 1.75 |
| `VOLT_VIOLATION_ALPHA` | 6.0 |

## Logging
Sparse epoch logging: `--eval_every 10`, `--log_every 0` (validation metrics every 10 epochs; no per-batch spam).

## Run suffixes
- Smoke (`SMOKE_TEST=True`): `_addons_moderate_smoke`
- Full (`SMOKE_TEST=False`, default): `_addons_moderate_full`

Data caches and checkpoints use the same Colab Drive paths as `nonunique.ipynb` (`/content/drive/MyDrive/datasets_gnn2/...`).


In [ ]:
# Clone or update GNN-Sandia repo (run first)
import os

REPO_DIR = "/content/GNN-Sandia"
REPO_URL = "https://github.com/alitasavori/GNN-Sandia.git"  # private? use a token URL

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull origin main

%cd $REPO_DIR
os.environ["GNN2_REPO_ROOT"] = REPO_DIR
print(f"Working directory: {os.getcwd()}")
!git log --oneline -3


In [ ]:
import os
import sys
import subprocess
import datetime
import warnings
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"
COLAB_CHUNK_DEFAULT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
COLAB_NODE_PE_DEFAULT = (
    COLAB_CHUNK_DEFAULT
    / "run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv"
)
COLAB_RUNS_PARENT = Path("/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints")
HOP_CSV_NAME = "load_hop_distance_to_each_regulator_all_index_nodes.csv"
WIN_CHUNK_DEFAULT = Path(r"D:\datasets\original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")


def _is_windows_drive_path(path) -> bool:
    s = str(path).strip().replace("/", "\\")
    return len(s) >= 2 and s[1] == ":" and s[0].isalpha()


def _on_colab() -> bool:
    return Path("/content").is_dir()


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _resolve_data_path(path: Path, *, label: str, colab_fallback: Path | None = None) -> Path:
    """Resolve a data path; never join Windows drive letters with cwd on Linux."""
    raw = str(path)
    if _is_windows_drive_path(raw):
        if _on_colab() and colab_fallback is not None:
            warnings.warn(
                f"{label}={raw!r} is a Windows path on Linux/Colab; using {colab_fallback} instead.",
                UserWarning,
                stacklevel=2,
            )
            return colab_fallback.expanduser().resolve()
        if _on_colab():
            raise ValueError(
                f"{label}={raw!r} is a Windows absolute path and invalid on Colab. "
                f"Mount Drive and use e.g. {COLAB_CHUNK_DEFAULT}"
            )
        return Path(raw).expanduser().resolve()
    p = Path(path).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (Path.cwd() / p).resolve()


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(
            f"No run_*/gnn_node_index_master.csv under {chunk_parent}"
        )
    return hits[0]


def _resolve_hop_csv(
    *,
    repo: Path,
    chunk_parent: Path,
    mydrive_data: Path | None,
) -> Path | None:
    """Resolve regulator hop CSV for --attn_reg_territory_bias (first existing candidate)."""
    env_raw = os.environ.get("GNN2_HOP_CSV", "").strip()
    candidates: list[Path] = []
    if env_raw:
        p = Path(env_raw).expanduser()
        candidates.append(p.resolve() if p.is_absolute() else (repo / p).resolve())
    if mydrive_data is not None:
        candidates.append((mydrive_data / HOP_CSV_NAME).resolve())
    cp = chunk_parent.resolve()
    candidates.extend(
        [
            (cp / ".." / HOP_CSV_NAME).resolve(),
            (cp.parent / HOP_CSV_NAME).resolve(),
        ]
    )
    candidates.extend(
        [
            (repo / "datasets_gnn2_from pc" / HOP_CSV_NAME).resolve(),
            (repo / "datasets_gnn2" / HOP_CSV_NAME).resolve(),
        ]
    )
    seen: set[Path] = set()
    for c in candidates:
        if c in seen:
            continue
        seen.add(c)
        if c.is_file():
            return c
    return None

def _sorted_run_chunk_dirs(chunk_parent: Path) -> list[Path]:
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )


def _smoke_chunk_subdir_glob(chunk_parent: Path, count: int) -> str:
    """First `count` run_* names, comma-separated for --chunk_subdir_glob."""
    names = [p.name for p in _sorted_run_chunk_dirs(chunk_parent)]
    if len(names) < count:
        raise ValueError(
            f"SMOKE_CHUNK_COUNT={count} but only {len(names)} run_* under {chunk_parent}"
        )
    return ",".join(names[:count])


def _chunks_from_subdir_glob(chunk_parent: Path, glob_pat: str) -> list[Path]:
    import fnmatch

    glob_pat = str(glob_pat).strip()
    if "," in glob_pat:
        allowed = {s.strip() for s in glob_pat.split(",") if s.strip()}
        chunks = sorted(
            (p for p in chunk_parent.iterdir() if p.is_dir() and p.name in allowed),
            key=lambda p: p.name,
        )
        missing = allowed - {p.name for p in chunks}
        if missing:
            raise FileNotFoundError(f"Missing smoke chunk folders: {sorted(missing)}")
        return chunks
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and fnmatch.fnmatch(p.name, glob_pat)),
        key=lambda p: p.name,
    )


def _write_smoke_compare_manifest(
    runs_parent: Path,
    *,
    out_dir: Path,
    physics_weight: float,
    chunk_glob: str,
    n_chunks: int,
    epochs: int,
    seed: int,
) -> None:
    import json as _json

    path = runs_parent / "_smoke_compare_manifest.json"
    manifest = {}
    if path.is_file():
        manifest = _json.loads(path.read_text(encoding="utf-8"))
    key = "physics" if physics_weight > 0 else "baseline"
    manifest[key] = {
        "run_dir": str(out_dir.resolve()),
        "physics_weight": physics_weight,
        "chunk_glob": chunk_glob,
        "n_chunks": n_chunks,
        "epochs": epochs,
        "seed": seed,
    }
    path.write_text(_json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Smoke compare manifest ({key}) -> {path}")


# Colab: mount Drive when chunks/caches live under MyDrive
if _on_colab() and not _drive_mounted():
    from google.colab import drive
    drive.mount("/content/drive")

# --- FULL training (set True only for quick smoke/debug) ---
SMOKE_TEST = False
SMOKE_CHUNK_COUNT = 3  # used only when SMOKE_TEST=True
SMOKE_EPOCHS = 15
SMOKE_PATIENCE = 5
SMOKE_SEED = 42
FULL_EPOCHS = 200
FULL_PATIENCE = 30
TRAIN_SEED = 42

# MODERATE add-on experiment (2026-07): current recommended config (best ep1 smoke performer).
# Hypothesis: middle-ground retune — stronger tap/attention than tuned baseline, partial violation up-weighting.
ATTN_REG_TERRITORY_BETA = 7.0   # was 6.0 tuned / 9.0 extreme / 2.0 default
REG_ORDINAL_ALPHA = 1.75        # was 1.25 tuned / 3.0 extreme / 1.0 default
VOLT_VIOLATION_ALPHA = 6.0      # was 8.0 tuned / 0.0 extreme (uniform MSE when 0)
_DA_CACHE_NAME = "da_gps_chunked_mvagg_smoke_gine" if SMOKE_TEST else "da_gps_chunked_mvagg_full_gine"

# --- paths: auto-detect Colab + Drive; override below if needed ---
_env_repo = os.environ.get("GNN2_REPO_ROOT", "").strip()
REPO = Path(_env_repo).expanduser().resolve() if _env_repo else Path.cwd().resolve()
if not (REPO / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
    raise FileNotFoundError(f"Repo root missing trainer script: {REPO}")

if _on_colab():
    if not _drive_mounted():
        raise RuntimeError(
            "Colab requires Google Drive mounted. Re-run this cell and approve Drive access, "
            "or: from google.colab import drive; drive.mount('/content/drive')"
        )
    CHUNK_PARENT = COLAB_CHUNK_DEFAULT
    NODE_PE_CSV = COLAB_NODE_PE_DEFAULT
    # Caches must live on Drive to reuse across Colab sessions (/content is wiped on reset)
    DA_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = COLAB_RUNS_PARENT
elif os.name == "nt":
    CHUNK_PARENT = WIN_CHUNK_DEFAULT
    NODE_PE_CSV = None  # auto: first run_*/gnn_node_index_master.csv under CHUNK_PARENT
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"
else:
    CHUNK_PARENT = REPO / "datasets_gnn2_from pc/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
    NODE_PE_CSV = None
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"

# Optional overrides (use POSIX paths on Colab, not D:\...):
# CHUNK_PARENT = Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")
# NODE_PE_CSV = CHUNK_PARENT / "run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv"
# RUNS_PARENT = MYDRIVE_DATA / "runs"  # persist checkpoints on Drive instead of ephemeral clone

# --- lighter architecture (locked in for add-ons full run) ---
LIGHTER_HIDDEN = 64
LIGHTER_LAYERS = 2
LIGHTER_HEADS = 2
LIGHTER_NODE_EMB_DIM = 2
LIGHTER_EDGE_EMB_DIM = 0

PHYSICS_WEIGHT = 0.0  # baseline; early_stop_on=total

# Explicit PF balance nodes when physics on (1177 hetero MV load nodes; chunk-safe)
PF_BALANCE_NODE_LIST_CSV = REPO / "colab_pf_data/pf_balance_nodes_explicit.csv"

META_AUX_COLS = "pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar"
NUM_WORKERS = 0 if os.name == "nt" else 4

os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

_colab_fb = COLAB_CHUNK_DEFAULT if _on_colab() else None
chunk_parent = _resolve_data_path(CHUNK_PARENT, label="CHUNK_PARENT", colab_fallback=_colab_fb)
if SMOKE_TEST:
    CHUNK_GLOB = _smoke_chunk_subdir_glob(chunk_parent, SMOKE_CHUNK_COUNT)
    EPOCHS = SMOKE_EPOCHS
    PATIENCE = SMOKE_PATIENCE
    SEED = SMOKE_SEED
else:
    CHUNK_GLOB = "run_*"
    EPOCHS = FULL_EPOCHS
    PATIENCE = FULL_PATIENCE
    SEED = TRAIN_SEED
da_cache_root = _resolve_data_path(
    DA_CACHE_ROOT,
    label="DA_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}" if _on_colab() else None,
)
gnn_cache_root = _resolve_data_path(
    GNN_CACHE_ROOT,
    label="GNN_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine" if _on_colab() else None,
)
runs_parent = _resolve_data_path(
    RUNS_PARENT,
    label="RUNS_PARENT",
    colab_fallback=MYDRIVE_DATA / "runs" if _on_colab() else None,
)

# --- PF topology root (repo colab_pf_data/ after git pull, or Drive dailyagg fallback) ---
from gnn2_pf_data_paths import PF_CAP_NODES_REL, PF_REG_CATALOG_REL, resolve_pf_catalog_paths

PF_DATA_ROOT = None
if PHYSICS_WEIGHT > 0:
    _reg_cat, _cap_map, PF_DATA_ROOT = resolve_pf_catalog_paths(
        repo=REPO,
        preferred_root=None,
        chunk_parent=chunk_parent,
    )

print("=== Preflight (add-ons MODERATE experiment) ===")
print(f"ADDON COEFFS:   territory_beta={ATTN_REG_TERRITORY_BETA} ordinal_alpha={REG_ORDINAL_ALPHA} volt_violation_alpha={VOLT_VIOLATION_ALPHA}")
print(f"REPO:           {REPO}")
print(f"SMOKE_TEST:     {SMOKE_TEST}")
print(f"SMOKE_CHUNK_COUNT: {SMOKE_CHUNK_COUNT}")
print(f"CHUNK_GLOB:     {CHUNK_GLOB}")
print(f"EPOCHS:         {EPOCHS}")
print(f"PATIENCE:       {PATIENCE}")
print(f"SEED:           {SEED}")
print(f"CHUNK_PARENT:   {chunk_parent}")
if not chunk_parent.is_dir():
    raise FileNotFoundError(f"CHUNK_PARENT not found: {chunk_parent}")

run_preview = sorted(
    p.name for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")
)[:5]
print(f"run_* preview:  {run_preview}")

if NODE_PE_CSV is None:
    node_pe = _find_node_pe_csv(chunk_parent)
else:
    node_pe = _resolve_data_path(NODE_PE_CSV, label="NODE_PE_CSV", colab_fallback=COLAB_NODE_PE_DEFAULT if _on_colab() else None)
    if not node_pe.is_file():
        warnings.warn(
            f"NODE_PE_CSV not found at {node_pe}; auto-discovering under CHUNK_PARENT.",
            UserWarning,
            stacklevel=2,
        )
        node_pe = _find_node_pe_csv(chunk_parent)

print(f"NODE_PE_CSV:    {node_pe}")
print(f"DA_CACHE_ROOT:  {da_cache_root}")
print(f"GNN_CACHE_ROOT: {gnn_cache_root}")
print(f"RUNS_PARENT:    {runs_parent}")

_mydrive_for_hop = MYDRIVE_DATA if _on_colab() and _drive_mounted() else None
hop_csv = _resolve_hop_csv(repo=REPO, chunk_parent=chunk_parent, mydrive_data=_mydrive_for_hop)
_hop_status = "OK" if hop_csv is not None else "MISSING"
print(f"HOP_CSV:        {_hop_status}" + (f"  {hop_csv}" if hop_csv else ""))
if hop_csv is None:
    _expected = MYDRIVE_DATA / HOP_CSV_NAME if _on_colab() else REPO / "datasets_gnn2_from pc" / HOP_CSV_NAME
    raise FileNotFoundError(
        "Regulator hop CSV required for --attn_reg_territory_bias but not found. "
        f"Generate with compute_hop_distance_all_index_nodes.py or copy to {_expected}"
    )
os.environ["GNN2_HOP_CSV"] = str(hop_csv)

from da_gps_hop_attention_ratios import TARGET_REG_COLS, validate_reg_hop_csv

_reg_csv = None
for _cand in (
    chunk_parent.parent / "regulator_involved_nodes.csv",
    REPO / "datasets_gnn2_from pc" / "loadtype_8500_dailyagg" / "regulator_involved_nodes.csv",
):
    if _cand.is_file():
        _reg_csv = _cand
        break
_hop_map = validate_reg_hop_csv(
    hop_csv,
    list(TARGET_REG_COLS),
    regulator_csv=_reg_csv,
)
print("HOP preflight (phase-consistent):", ", ".join(f"{k}->{v}" for k, v in _hop_map.items()))

# Phase-consistent hop column mapping preflight (territory bias is harmful if wrong).
import pandas as pd
from da_gps_hop_attention_ratios import REG_COL_TO_HOP_COL, validate_reg_hop_csv
from train_da_gps_multitask_complex_voltage import TARGET_REG_COLS

_hop_node_names = pd.read_csv(node_pe)["node"].astype(str).tolist()
_reg_csv_candidates = [
    chunk_parent.parent / "regulator_involved_nodes.csv",
    REPO / "datasets_gnn2_from pc" / "loadtype_8500_dailyagg" / "regulator_involved_nodes.csv",
]
_hop_reg_csv = next((p for p in _reg_csv_candidates if p.is_file()), None)
_hop_pairs = validate_reg_hop_csv(
    hop_csv,
    list(TARGET_REG_COLS),
    node_names=_hop_node_names,
    reg_col_to_hop_col=REG_COL_TO_HOP_COL,
    regulator_csv=_hop_reg_csv,
)
print("HOP mapping preflight (reg_col -> hop_col):")
for _rc, _hc in _hop_pairs.items():
    print(f"  {_rc} -> {_hc}")

if PHYSICS_WEIGHT > 0:
    assert PF_DATA_ROOT is not None
    _pf_checks = [
        ("reg_catalog", PF_DATA_ROOT / PF_REG_CATALOG_REL),
        ("cap_nodes", PF_DATA_ROOT / PF_CAP_NODES_REL),
        ("electrical_distance", PF_DATA_ROOT / "electrical_distance_from_substation.csv"),
        (
            "hetero_mv_nodes",
            PF_DATA_ROOT / "Heterogenous GNN dataset/nodes/hetero_mv_nodes_load_transformer.csv",
        ),
        ("bus_kv_cache", PF_DATA_ROOT / "bus_kv_base_by_node.csv"),
    ]
    print(f"PF_DATA_ROOT:   {PF_DATA_ROOT}")
    for label, p in _pf_checks:
        status = "OK" if p.is_file() else "MISSING"
        print(f"  PF {label}: {status}  {p}")
        if not p.is_file():
            raise FileNotFoundError(f"Physics preflight missing {label}: {p}")
else:
    print("PF_DATA_ROOT:   (skipped — PHYSICS_WEIGHT=0)")

print("=================")

da_cache_root.mkdir(parents=True, exist_ok=True)
gnn_cache_root.mkdir(parents=True, exist_ok=True)
runs_parent.mkdir(parents=True, exist_ok=True)

chunks = _chunks_from_subdir_glob(chunk_parent, CHUNK_GLOB)
if not chunks:
    raise RuntimeError(f"No folders for CHUNK_GLOB={CHUNK_GLOB!r} under {chunk_parent}")

for p in chunks:
    for name in (
        "gnn_node_features_and_targets_mvagg.csv",
        "gnn_edges_phase_static.csv",
        "gnn_sample_meta.csv",
    ):
        if not (p / name).is_file():
            raise FileNotFoundError(f"Missing {name} in {p.name}")

print(f"Found {len(chunks)} chunk(s) for CHUNK_GLOB under {chunk_parent}")

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_pf_suffix = "_pf" if PHYSICS_WEIGHT > 0 else ""
_addon_run_suffix = "_addons_moderate_smoke" if SMOKE_TEST else "_addons_moderate_full"
out_dir = runs_parent / (
    f"da_gps_chunked_l{LIGHTER_LAYERS}_h{LIGHTER_HIDDEN}_mvagg_gine_metaaux_regce{_pf_suffix}{_addon_run_suffix}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

_early_stop = "voltage" if PHYSICS_WEIGHT > 0 else "total"

cmd = [
    sys.executable, "-u", "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent", str(chunk_parent),
    "--chunk_subdir_glob", CHUNK_GLOB,
    "--nodes_csv", "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv", "gnn_edges_phase_static.csv",
    "--meta_csv", "gnn_sample_meta.csv",
    "--node_feature_cols", "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv", str(node_pe),
    "--node_pe_cols", "auto",
    "--n_system_tokens", "10",
    "--aux_meta_cols", META_AUX_COLS,
    "--lambda_pv", "0.1",
    "--out_dir", str(out_dir),
    "--cache_dir", str(da_cache_root),
    "--bootstrap_gnn_cache_dir", str(gnn_cache_root),
    "--epochs", str(EPOCHS),
    "--batch_size", "64",
    "--hidden", str(LIGHTER_HIDDEN),
    "--layers", str(LIGHTER_LAYERS),
    "--heads", str(LIGHTER_HEADS),
    "--node_emb_dim", str(LIGHTER_NODE_EMB_DIM),
    "--edge_emb_dim", str(LIGHTER_EDGE_EMB_DIM),
    "--lr", "5e-4",
    "--weight_decay", "1e-5",
    "--lambda_cap", "0.1",
    "--lambda_reg", "0.1",
    "--reg_loss", "ce",
    "--per_device_cap_head",
    "--per_device_reg_head",
    "--patience", str(PATIENCE),
    "--seed", str(SEED),
    "--train_frac", "0.80",
    "--val_frac", "0.10",
    "--sample_frac", "1.0",
    "--num_workers", str(NUM_WORKERS),
    "--eval_every", "10",
    "--log_every", "0",
    "--checkpoint_every", "10",
    "--early_stop_on", _early_stop,
    "--dropout", "0.1",
    "--hop_csv", str(hop_csv),
    "--attn_reg_territory_bias",
    "--attn_reg_territory_beta", str(ATTN_REG_TERRITORY_BETA),
    "--reg_hybrid_tap_loss",
    "--reg_ordinal_alpha", str(REG_ORDINAL_ALPHA),
    "--volt_violation_weight",
    "--volt_lo_pu", "0.96",
    "--volt_hi_pu", "1.04",
    "--volt_violation_alpha", str(VOLT_VIOLATION_ALPHA),
]

if PHYSICS_WEIGHT > 0:
    _pf_flags = [
        "--pf_data_root", str(PF_DATA_ROOT),
        "--loss_power_balance_weight", str(PHYSICS_WEIGHT),
        "--pf_sparse_y", "1",
        "--pf_huber_delta_kw", "10",
        "--pf_detach_controls",
    ]
    _bal = Path(PF_BALANCE_NODE_LIST_CSV)
    if not _bal.is_absolute():
        _bal = (REPO / _bal).resolve()
    if not _bal.is_file():
        raise FileNotFoundError(f"PF_BALANCE_NODE_LIST_CSV not found: {_bal}")
    _pf_flags.extend(["--pf_balance_node_list_csv", str(_bal)])
    cmd.extend(_pf_flags)

_mode = "physics-informed" if PHYSICS_WEIGHT > 0 else "baseline"
print(f"PHYSICS_WEIGHT={PHYSICS_WEIGHT} ({_mode})")
print(f"early_stop_on={_early_stop}")
print("\nRunning:\n ", " ".join(cmd), "\n", flush=True)

with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nTraining completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Regulator classes:", (out_dir / "reg_class_tables.json").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())
_run_name = out_dir.name
print("\n=== Section 8 prerequisite ===")
print(f"Baseline run folder:  {out_dir.resolve()}")
print(f"Section 8 expects:    {MYDRIVE_DATA / 'runs' / _run_name}")
print("Checkpoint for init:  da_gps_multitask_best.pt (or training_last.pt)")
if _on_colab() and not str(out_dir.resolve()).startswith(str(MYDRIVE_DATA.resolve())):
    print(
        "WARNING: run dir is NOT under MyDrive/datasets_gnn2 — "
        "copy to Drive before disconnecting the VM or Section 8 will fail."
    )
else:
    print("Checkpoints are on Drive; safe to run Section 8 after this session ends.")
_write_smoke_compare_manifest(
    runs_parent,
    out_dir=out_dir,
    physics_weight=PHYSICS_WEIGHT,
    chunk_glob=CHUNK_GLOB,
    n_chunks=len(chunks),
    epochs=EPOCHS,
    seed=SEED,
)


=== Preflight (add-ons MODERATE experiment) ===
ADDON COEFFS:   territory_beta=7.0 ordinal_alpha=1.75 volt_violation_alpha=6.0
REPO:           /content/GNN-Sandia
SMOKE_TEST:     False
SMOKE_CHUNK_COUNT: 3
CHUNK_GLOB:     run_*
EPOCHS:         200
PATIENCE:       30
SEED:           42
CHUNK_PARENT:   /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40
run_* preview:  ['run_001_scen_0000_0049_seed_20420233', 'run_002_scen_0050_0099_seed_20520236', 'run_003_scen_0100_0149_seed_20620239', 'run_004_scen_0150_0199_seed_20720242', 'run_005_scen_0200_0249_seed_20820245']
NODE_PE_CSV:    /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv
DA_CACHE_ROOT:  /content/drive/MyDrive/datasets_gnn2/cache/da_gps_chunked_mvagg_full_gine
GNN_CACHE_ROOT: /content/drive/MyDrive/datasets_gnn2/cache/gnn_only_chunked_mvagg_full_gine
RUNS_PARENT:    /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints
HOP_CSV:        OK  /content/drive/MyDrive/datasets_gnn2/load_hop_distance_to_each_regulator_all_index_nodes.csv
HOP preflight (phase-consistent): reg_feeder_rega_tap_pu->FEEDER_REGA, reg_feeder_regb_tap_pu->FEEDER_REGB, reg_feeder_regc_tap_pu->FEEDER_REGC, reg_vreg2_a_tap_pu->VREG2_A, reg_vreg2_b_tap_pu->VREG2_B, reg_vreg2_c_tap_pu->VREG2_C, reg_vreg3_a_tap_pu->VREG3_A, reg_vreg3_b_tap_pu->VREG3_B, reg_vreg3_c_tap_pu->VREG3_C, reg_vreg4_a_tap_pu->VREG4_A, reg_vreg4_b_tap_pu->VREG4_B, reg_vreg4_c_tap_pu->VREG4_C
HOP mapping preflight (reg_col -> hop_col):
  reg_feeder_rega_tap_pu -> FEEDER_REGA
  reg_feeder_regb_tap_pu -> FEEDER_REGB
  reg_feeder_regc_tap_pu -> FEEDER_REGC
  reg_vreg2_a_tap_pu -> VREG2_A
  reg_vreg2_b_tap_pu -> VREG2_B
  reg_vreg2_c_tap_pu -> VREG2_C
  reg_vreg3_a_tap_pu -> VREG3_A
  reg_vreg3_b_tap_pu -> VREG3_B
  reg_vreg3_c_tap_pu -> VREG3_C
  reg_vreg4_a_tap_pu -> VREG4_A
  reg_vreg4_b_tap_pu -> VREG4_B
  reg_vreg4_c_tap_pu -> VREG4_C
PF_DATA_ROOT:   (skipped — PHYSICS_WEIGHT=0)
=================
Found 40 chunk(s) for CHUNK_GLOB under /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40
PHYSICS_WEIGHT=0.0 (baseline)
early_stop_on=total

Running:
  /usr/bin/python3 -u train_da_gps_multitask_complex_voltage_gine.py --chunk_parent /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40 --chunk_subdir_glob run_* --nodes_csv gnn_node_features_and_targets_mvagg.csv --edge_catalog_csv gnn_edges_phase_static.csv --meta_csv gnn_sample_meta.csv --node_feature_cols p_load_kw,q_load_kvar,p_pv_kw --exclude_bess_features --node_pe_csv /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv --node_pe_cols auto --n_system_tokens 10 --aux_meta_cols pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar --lambda_pv 0.1 --out_dir /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401 --cache_dir /content/drive/MyDrive/datasets_gnn2/cache/da_gps_chunked_mvagg_full_gine --bootstrap_gnn_cache_dir /content/drive/MyDrive/datasets_gnn2/cache/gnn_only_chunked_mvagg_full_gine --epochs 200 --batch_size 64 --hidden 64 --layers 2 --heads 2 --node_emb_dim 2 --edge_emb_dim 0 --lr 5e-4 --weight_decay 1e-5 --lambda_cap 0.1 --lambda_reg 0.1 --reg_loss ce --per_device_cap_head --per_device_reg_head --patience 30 --seed 42 --train_frac 0.80 --val_frac 0.10 --sample_frac 1.0 --num_workers 4 --eval_every 10 --log_every 0 --checkpoint_every 10 --early_stop_on total --dropout 0.1 --hop_csv /content/drive/MyDrive/datasets_gnn2/load_hop_distance_to_each_regulator_all_index_nodes.csv --attn_reg_territory_bias --attn_reg_territory_beta 7.0 --reg_hybrid_tap_loss --reg_ordinal_alpha 1.75 --volt_violation_weight --volt_lo_pu 0.96 --volt_hi_pu 1.04 --volt_violation_alpha 6.0 

regulator tap training loss: ce (discrete tap classes + cross-entropy)
exclude_bess_features: using node_feature_cols= ['p_load_kw', 'q_load_kvar', 'p_pv_kw']
chunk_parent cache override via --cache_dir: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_chunked_mvagg_full_gine
bootstrap GNN cache dir: /content/drive/MyDrive/datasets_gnn2/cache/gnn_only_chunked_mvagg_full_gine
Meta aux (sample_meta): 4 column(s); chunk DA caches use suffix __mauxb7bd1d58 (per chunk name).
  global token index 22 (system slot 0): column 'pv_pv2_p_post_kw'
  global token index 23 (system slot 1): column 'pv_pv2_q_post_kvar'
  global token index 24 (system slot 2): column 'p_loss_total_post_kw'
  global token index 25 (system slot 3): column 'q_loss_total_post_kvar'
[chunk_parent] 40 chunks under /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40
  - run_001_scen_0000_0049_seed_20420233
  - run_002_scen_0050_0099_seed_20520236
  - run_003_scen_0100_0149_seed_20620239
  - run_004_scen_0150_0199_seed_20720242
  - run_005_scen_0200_0249_seed_20820245
  - run_006_scen_0250_0299_seed_20920248
  - run_007_scen_0300_0349_seed_21020251
  - run_008_scen_0350_0399_seed_21120254
  - run_009_scen_0400_0449_seed_21220257
  - run_010_scen_0450_0499_seed_21320260
  - run_011_scen_0500_0549_seed_21420263
  - run_012_scen_0550_0599_seed_21520266
  - run_013_scen_0600_0649_seed_21620269
  - run_014_scen_0650_0699_seed_21720272
  - run_015_scen_0700_0749_seed_21820275
  - run_016_scen_0750_0799_seed_21920278
  - run_017_scen_0800_0849_seed_22020281
  - run_018_scen_0850_0899_seed_22120284
  - run_019_scen_0900_0949_seed_22220287
  - run_020_scen_0950_0999_seed_22320290
  - run_021_scen_1000_1049_seed_22420293
  - run_022_scen_1050_1099_seed_22520296
  - run_023_scen_1100_1149_seed_22620299
  - run_024_scen_1150_1199_seed_22720302
  - run_025_scen_1200_1249_seed_22820305
  - run_026_scen_1250_1299_seed_22920308
  - run_027_scen_1300_1349_seed_23020311
  - run_028_scen_1350_1399_seed_23120314
  - run_029_scen_1400_1449_seed_23220317
  - run_030_scen_1450_1499_seed_23320320
  - run_031_scen_1500_1549_seed_23420323
  - run_032_scen_1550_1599_seed_23520326
  - run_033_scen_1600_1649_seed_23620329
  - run_034_scen_1650_1699_seed_23720332
  - run_035_scen_1700_1749_seed_23820335
  - run_036_scen_1750_1799_seed_23920338
  - run_037_scen_1800_1849_seed_24020341
  - run_038_scen_1850_1899_seed_24120344
  - run_039_scen_1900_1949_seed_24220347
  - run_040_scen_1950_1999_seed_24320350
reg_loss=ce: per-regulator tap classes (rounded unique tap_pu):
  reg_feeder_rega_tap_pu: n_classes=19
  reg_feeder_regb_tap_pu: n_classes=19
  reg_feeder_regc_tap_pu: n_classes=19
  reg_vreg2_a_tap_pu: n_classes=19
  reg_vreg2_b_tap_pu: n_classes=22
  reg_vreg2_c_tap_pu: n_classes=29
  reg_vreg3_a_tap_pu: n_classes=21
  reg_vreg3_b_tap_pu: n_classes=22
  reg_vreg3_c_tap_pu: n_classes=24
  reg_vreg4_a_tap_pu: n_classes=21
  reg_vreg4_b_tap_pu: n_classes=22
  reg_vreg4_c_tap_pu: n_classes=21
Loading compacted edges: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
  directed edges: 15256
norm stats: computed from current train split
Wrote run manifest (for daily compare / mid-train snapshots): /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/da_gps_run_manifest.json
reg_hybrid_tap_loss: enabled (CE + alpha * normalized expected ordinal tap cost, alpha=1.75)
torch.compile: enabled
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:6325: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = _GradScaler()
AMP (autocast + GradScaler): enabled
attn_reg_territory_bias: reg_col->hop_col mapping: reg_feeder_rega_tap_pu->FEEDER_REGA, reg_feeder_regb_tap_pu->FEEDER_REGB, reg_feeder_regc_tap_pu->FEEDER_REGC, reg_vreg2_a_tap_pu->VREG2_A, reg_vreg2_b_tap_pu->VREG2_B, reg_vreg2_c_tap_pu->VREG2_C, reg_vreg3_a_tap_pu->VREG3_A, reg_vreg3_b_tap_pu->VREG3_B, reg_vreg3_c_tap_pu->VREG3_C, reg_vreg4_a_tap_pu->VREG4_A, reg_vreg4_b_tap_pu->VREG4_B, reg_vreg4_c_tap_pu->VREG4_C
attn_reg_territory_bias: hop_csv=/content/drive/MyDrive/datasets_gnn2/load_hop_distance_to_each_regulator_all_index_nodes.csv rule='hop_gt_0' beta=7 downstream_frac=0.1795
[da_gps addons] expected relative scales at setup:
  territory: beta=7 on downstream node×reg keys (~18%% active pairs); typical scaled dot-product logits ~O(0.18) per head (d_head=32); beta adds up to +7 before softmax
  hybrid tap: CE ~1-5 typical; ordinal cost normalized to [0,1] per regulator (raw tap spans 0.1125-0.1750 pu); alpha=1.75 → ordinal term ~0.14-0.26 (~8-15% of plain CE target); effective in total loss: lambda_reg=0.1×(CE+1.75×ordinal) → ordinal contributes ~0.014-0.026 to val_tot
  voltage: mode=continuous w=1+6*depth outside [0.96,1.04]; at viol_frac~0.30, depth~0.01-0.03 pu → w_mean~1.5-2.5 (stronger near limits); lambda_v=1 scales weighted MSE in total loss
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:6520: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:6520: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:6733: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:6880: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
[da_gps chunk_parent] epoch    1/200 | train_tot=0.3919 train_volt=0.1554 train_cap=0.2221 train_reg=1.8214 train_meta_aux=0.3225 val_meta_aux=0.2799 | val_tot=0.2469 val_volt=0.0642 val_cap=0.1463 val_reg=1.4004 | val_r2_mean=0.8317 val_r2_min=-1.7269 val_r2_min_node=m1209814.1 val_worst_mae=0.0316 | best=0.2469
[da_gps chunk_parent addons] epoch 1 | territory_frac=0.1795 | train plain_ce=1.6376 hybrid_total=1.8213 ordinal_cost=0.1837 ordinal_frac=10.7% reg_tap_mae_pu=0.0085 reg_tap_acc=0.4032 | volt loss_w=0.1552 loss_u=0.1489 delta=0.0063 | val_reg=1.4001 val_tap_mae=0.0060 val_tap_acc=0.4395 val_volt_uniform=0.0642 val_hybrid_total=1.5135 val_plain_ce=1.4001 val_ordinal_cost=0.1134 val_ordinal_frac=8.1% | val_tot=0.2469 [1*volt(0.0642)+0.1*cap(0.1463)+0.1*reg(1.4004)+0.1*meta(0.2799)]
[da_gps addons counterfactual] epoch 1 | territory: val_reg=1.4001 no_territory=1.5382 delta=-0.1381 (negative helps) | hybrid: val_plain_ce=1.4001 val_ordinal_cost=0.1134 val_hybrid_total=1.5135 delta_hybrid_vs_plain=0.1134 (positive hurts val_reg if hybrid used) ordinal_frac=8.1% val_tap_acc=0.4395 | voltage: val_uniform=0.0642 val_weighted=0.0652 viol_frac=0.2970 delta=0.0010
[da_gps chunk_parent] epoch   10/200 | train_tot=0.1644 train_volt=0.0339 train_cap=0.0887 train_reg=1.2011 train_meta_aux=0.0158 val_meta_aux=0.0107 | val_tot=0.1498 val_volt=0.0303 val_cap=0.0846 val_reg=1.0993 | val_r2_mean=0.8831 val_r2_min=-0.8489 val_r2_min_node=_hvmv_sub_lsb.2 val_worst_mae=0.0229 | best=0.1498
[da_gps chunk_parent addons] epoch 10 | territory_frac=0.1795 | train plain_ce=1.1234 hybrid_total=1.2007 ordinal_cost=0.0773 ordinal_frac=6.9% reg_tap_mae_pu=0.0043 reg_tap_acc=0.5324 | volt loss_w=0.0339 loss_u=0.0334 delta=0.0004 | val_reg=1.0991 val_tap_mae=0.0042 val_tap_acc=0.5408 val_volt_uniform=0.0303 val_hybrid_total=1.1744 val_plain_ce=1.0991 val_ordinal_cost=0.0753 val_ordinal_frac=6.9% | val_tot=0.1498 [1*volt(0.0303)+0.1*cap(0.0846)+0.1*reg(1.0993)+0.1*meta(0.0107)]
[da_gps addons counterfactual] epoch 10 | territory: val_reg=1.0991 no_territory=1.3634 delta=-0.2643 (negative helps) | hybrid: val_plain_ce=1.0991 val_ordinal_cost=0.0753 val_hybrid_total=1.1744 delta_hybrid_vs_plain=0.0753 (positive hurts val_reg if hybrid used) ordinal_frac=6.9% val_tap_acc=0.5408 | voltage: val_uniform=0.0303 val_weighted=0.0305 viol_frac=0.2970 delta=0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch   20/200 | train_tot=0.1537 train_volt=0.0294 train_cap=0.0804 train_reg=1.1509 train_meta_aux=0.0121 val_meta_aux=0.0080 | val_tot=0.1425 val_volt=0.0272 val_cap=0.0755 val_reg=1.0699 | val_r2_mean=0.8910 val_r2_min=-0.0360 val_r2_min_node=e192860.3 val_worst_mae=0.0217 | best=0.1425
[da_gps chunk_parent addons] epoch 20 | territory_frac=0.1795 | train plain_ce=1.0778 hybrid_total=1.1506 ordinal_cost=0.0728 ordinal_frac=6.8% reg_tap_mae_pu=0.0040 reg_tap_acc=0.5507 | volt loss_w=0.0294 loss_u=0.0291 delta=0.0004 | val_reg=1.0697 val_tap_mae=0.0040 val_tap_acc=0.5497 val_volt_uniform=0.0272 val_hybrid_total=1.1414 val_plain_ce=1.0697 val_ordinal_cost=0.0717 val_ordinal_frac=6.7% | val_tot=0.1425 [1*volt(0.0272)+0.1*cap(0.0755)+0.1*reg(1.0699)+0.1*meta(0.0080)]
[da_gps addons counterfactual] epoch 20 | territory: val_reg=1.0697 no_territory=1.4306 delta=-0.3609 (negative helps) | hybrid: val_plain_ce=1.0697 val_ordinal_cost=0.0717 val_hybrid_total=1.1414 delta_hybrid_vs_plain=0.0717 (positive hurts val_reg if hybrid used) ordinal_frac=6.7% val_tap_acc=0.5497 | voltage: val_uniform=0.0272 val_weighted=0.0277 viol_frac=0.2970 delta=0.0005
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch   30/200 | train_tot=0.1368 train_volt=0.0209 train_cap=0.0768 train_reg=1.0711 train_meta_aux=0.0108 val_meta_aux=0.0070 | val_tot=0.1255 val_volt=0.0190 val_cap=0.0728 val_reg=0.9862 | val_r2_mean=0.9138 val_r2_min=-0.0506 val_r2_min_node=190-8593.3 val_worst_mae=0.0192 | best=0.1255
[da_gps chunk_parent addons] epoch 30 | territory_frac=0.1795 | train plain_ce=1.0067 hybrid_total=1.0713 ordinal_cost=0.0646 ordinal_frac=6.4% reg_tap_mae_pu=0.0035 reg_tap_acc=0.5724 | volt loss_w=0.0209 loss_u=0.0208 delta=0.0001 | val_reg=0.9860 val_tap_mae=0.0035 val_tap_acc=0.5777 val_volt_uniform=0.0190 val_hybrid_total=1.0490 val_plain_ce=0.9860 val_ordinal_cost=0.0630 val_ordinal_frac=6.4% | val_tot=0.1255 [1*volt(0.0190)+0.1*cap(0.0728)+0.1*reg(0.9862)+0.1*meta(0.0070)]
[da_gps addons counterfactual] epoch 30 | territory: val_reg=0.9860 no_territory=2.8154 delta=-1.8294 (negative helps) | hybrid: val_plain_ce=0.9860 val_ordinal_cost=0.0630 val_hybrid_total=1.0490 delta_hybrid_vs_plain=0.0630 (positive hurts val_reg if hybrid used) ordinal_frac=6.4% val_tap_acc=0.5777 | voltage: val_uniform=0.0190 val_weighted=0.0189 viol_frac=0.2970 delta=-0.0001
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch   40/200 | train_tot=0.1287 train_volt=0.0175 train_cap=0.0735 train_reg=1.0288 train_meta_aux=0.0098 val_meta_aux=0.0086 | val_tot=0.1314 val_volt=0.0274 val_cap=0.0702 val_reg=0.9611 | val_r2_mean=0.8931 val_r2_min=-0.4671 val_r2_min_node=190-8593.3 val_worst_mae=0.0201 | best=0.1255
[da_gps chunk_parent addons] epoch 40 | territory_frac=0.1795 | train plain_ce=0.9681 hybrid_total=1.0286 ordinal_cost=0.0605 ordinal_frac=6.3% reg_tap_mae_pu=0.0033 reg_tap_acc=0.5857 | volt loss_w=0.0174 loss_u=0.0174 delta=0.0000 | val_reg=0.9611 val_tap_mae=0.0033 val_tap_acc=0.5895 val_volt_uniform=0.0275 val_hybrid_total=1.0207 val_plain_ce=0.9611 val_ordinal_cost=0.0596 val_ordinal_frac=6.2% | val_tot=0.1314 [1*volt(0.0274)+0.1*cap(0.0702)+0.1*reg(0.9611)+0.1*meta(0.0086)]
[da_gps addons counterfactual] epoch 40 | territory: val_reg=0.9611 no_territory=3.4125 delta=-2.4514 (negative helps) | hybrid: val_plain_ce=0.9611 val_ordinal_cost=0.0596 val_hybrid_total=1.0207 delta_hybrid_vs_plain=0.0596 (positive hurts val_reg if hybrid used) ordinal_frac=6.2% val_tap_acc=0.5895 | voltage: val_uniform=0.0275 val_weighted=0.0272 viol_frac=0.2970 delta=-0.0003
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch   50/200 | train_tot=0.1252 train_volt=0.0163 train_cap=0.0715 train_reg=1.0080 train_meta_aux=0.0093 val_meta_aux=0.0080 | val_tot=0.1120 val_volt=0.0133 val_cap=0.0692 val_reg=0.9099 | val_r2_mean=0.9256 val_r2_min=-0.1239 val_r2_min_node=190-8593.3 val_worst_mae=0.0167 | best=0.1120
[da_gps chunk_parent addons] epoch 50 | territory_frac=0.1795 | train plain_ce=0.9492 hybrid_total=1.0079 ordinal_cost=0.0587 ordinal_frac=6.2% reg_tap_mae_pu=0.0032 reg_tap_acc=0.5930 | volt loss_w=0.0163 loss_u=0.0163 delta=-0.0000 | val_reg=0.9100 val_tap_mae=0.0030 val_tap_acc=0.6075 val_volt_uniform=0.0133 val_hybrid_total=0.9658 val_plain_ce=0.9100 val_ordinal_cost=0.0558 val_ordinal_frac=6.1% | val_tot=0.1120 [1*volt(0.0133)+0.1*cap(0.0692)+0.1*reg(0.9099)+0.1*meta(0.0080)]
[da_gps addons counterfactual] epoch 50 | territory: val_reg=0.9100 no_territory=3.6787 delta=-2.7687 (negative helps) | hybrid: val_plain_ce=0.9100 val_ordinal_cost=0.0558 val_hybrid_total=0.9658 delta_hybrid_vs_plain=0.0558 (positive hurts val_reg if hybrid used) ordinal_frac=6.1% val_tap_acc=0.6075 | voltage: val_uniform=0.0133 val_weighted=0.0131 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch   60/200 | train_tot=0.1217 train_volt=0.0149 train_cap=0.0701 train_reg=0.9897 train_meta_aux=0.0085 val_meta_aux=0.0071 | val_tot=0.1151 val_volt=0.0140 val_cap=0.0702 val_reg=0.9332 | val_r2_mean=0.9258 val_r2_min=-0.1209 val_r2_min_node=190-8593.3 val_worst_mae=0.0164 | best=0.1120
[da_gps chunk_parent addons] epoch 60 | territory_frac=0.1795 | train plain_ce=0.9326 hybrid_total=0.9898 ordinal_cost=0.0572 ordinal_frac=6.1% reg_tap_mae_pu=0.0031 reg_tap_acc=0.5992 | volt loss_w=0.0149 loss_u=0.0149 delta=-0.0001 | val_reg=0.9333 val_tap_mae=0.0031 val_tap_acc=0.5980 val_volt_uniform=0.0140 val_hybrid_total=0.9896 val_plain_ce=0.9333 val_ordinal_cost=0.0564 val_ordinal_frac=6.0% | val_tot=0.1151 [1*volt(0.0140)+0.1*cap(0.0702)+0.1*reg(0.9332)+0.1*meta(0.0071)]
[da_gps addons counterfactual] epoch 60 | territory: val_reg=0.9333 no_territory=4.2390 delta=-3.3057 (negative helps) | hybrid: val_plain_ce=0.9333 val_ordinal_cost=0.0564 val_hybrid_total=0.9896 delta_hybrid_vs_plain=0.0564 (positive hurts val_reg if hybrid used) ordinal_frac=6.0% val_tap_acc=0.5980 | voltage: val_uniform=0.0140 val_weighted=0.0139 viol_frac=0.2970 delta=-0.0001
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch   70/200 | train_tot=0.1200 train_volt=0.0144 train_cap=0.0691 train_reg=0.9786 train_meta_aux=0.0083 val_meta_aux=0.0069 | val_tot=0.1114 val_volt=0.0149 val_cap=0.0701 val_reg=0.8882 | val_r2_mean=0.9232 val_r2_min=-0.1769 val_r2_min_node=190-8593.3 val_worst_mae=0.0160 | best=0.1114
[da_gps chunk_parent addons] epoch 70 | territory_frac=0.1795 | train plain_ce=0.9221 hybrid_total=0.9784 ordinal_cost=0.0563 ordinal_frac=6.1% reg_tap_mae_pu=0.0031 reg_tap_acc=0.6026 | volt loss_w=0.0144 loss_u=0.0145 delta=-0.0001 | val_reg=0.8883 val_tap_mae=0.0029 val_tap_acc=0.6184 val_volt_uniform=0.0149 val_hybrid_total=0.9424 val_plain_ce=0.8883 val_ordinal_cost=0.0541 val_ordinal_frac=6.1% | val_tot=0.1114 [1*volt(0.0149)+0.1*cap(0.0701)+0.1*reg(0.8882)+0.1*meta(0.0069)]
[da_gps addons counterfactual] epoch 70 | territory: val_reg=0.8883 no_territory=4.0379 delta=-3.1495 (negative helps) | hybrid: val_plain_ce=0.8883 val_ordinal_cost=0.0541 val_hybrid_total=0.9424 delta_hybrid_vs_plain=0.0541 (positive hurts val_reg if hybrid used) ordinal_frac=6.1% val_tap_acc=0.6184 | voltage: val_uniform=0.0149 val_weighted=0.0150 viol_frac=0.2970 delta=0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch   80/200 | train_tot=0.1182 train_volt=0.0138 train_cap=0.0681 train_reg=0.9679 train_meta_aux=0.0080 val_meta_aux=0.0070 | val_tot=0.1093 val_volt=0.0124 val_cap=0.0662 val_reg=0.8957 | val_r2_mean=0.9281 val_r2_min=-0.0096 val_r2_min_node=190-8593.3 val_worst_mae=0.0159 | best=0.1093
[da_gps chunk_parent addons] epoch 80 | territory_frac=0.1795 | train plain_ce=0.9122 hybrid_total=0.9676 ordinal_cost=0.0554 ordinal_frac=6.1% reg_tap_mae_pu=0.0030 reg_tap_acc=0.6064 | volt loss_w=0.0138 loss_u=0.0139 delta=-0.0001 | val_reg=0.8958 val_tap_mae=0.0029 val_tap_acc=0.6130 val_volt_uniform=0.0125 val_hybrid_total=0.9493 val_plain_ce=0.8958 val_ordinal_cost=0.0535 val_ordinal_frac=6.0% | val_tot=0.1093 [1*volt(0.0124)+0.1*cap(0.0662)+0.1*reg(0.8957)+0.1*meta(0.0070)]
[da_gps addons counterfactual] epoch 80 | territory: val_reg=0.8958 no_territory=3.9889 delta=-3.0931 (negative helps) | hybrid: val_plain_ce=0.8958 val_ordinal_cost=0.0535 val_hybrid_total=0.9493 delta_hybrid_vs_plain=0.0535 (positive hurts val_reg if hybrid used) ordinal_frac=6.0% val_tap_acc=0.6130 | voltage: val_uniform=0.0125 val_weighted=0.0123 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch   90/200 | train_tot=0.1170 train_volt=0.0134 train_cap=0.0670 train_reg=0.9612 train_meta_aux=0.0079 val_meta_aux=0.0062 | val_tot=0.1083 val_volt=0.0131 val_cap=0.0643 val_reg=0.8819 | val_r2_mean=0.9279 val_r2_min=-0.0633 val_r2_min_node=190-8593.3 val_worst_mae=0.0152 | best=0.1083
[da_gps chunk_parent addons] epoch 90 | territory_frac=0.1795 | train plain_ce=0.9062 hybrid_total=0.9611 ordinal_cost=0.0549 ordinal_frac=6.1% reg_tap_mae_pu=0.0030 reg_tap_acc=0.6090 | volt loss_w=0.0134 loss_u=0.0135 delta=-0.0001 | val_reg=0.8820 val_tap_mae=0.0029 val_tap_acc=0.6188 val_volt_uniform=0.0131 val_hybrid_total=0.9348 val_plain_ce=0.8820 val_ordinal_cost=0.0528 val_ordinal_frac=6.0% | val_tot=0.1083 [1*volt(0.0131)+0.1*cap(0.0643)+0.1*reg(0.8819)+0.1*meta(0.0062)]
[da_gps addons counterfactual] epoch 90 | territory: val_reg=0.8820 no_territory=3.3139 delta=-2.4318 (negative helps) | hybrid: val_plain_ce=0.8820 val_ordinal_cost=0.0528 val_hybrid_total=0.9348 delta_hybrid_vs_plain=0.0528 (positive hurts val_reg if hybrid used) ordinal_frac=6.0% val_tap_acc=0.6188 | voltage: val_uniform=0.0131 val_weighted=0.0130 viol_frac=0.2970 delta=-0.0001
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  100/200 | train_tot=0.1160 train_volt=0.0132 train_cap=0.0663 train_reg=0.9544 train_meta_aux=0.0077 val_meta_aux=0.0068 | val_tot=0.1085 val_volt=0.0123 val_cap=0.0665 val_reg=0.8894 | val_r2_mean=0.9290 val_r2_min=0.0414 val_r2_min_node=190-8593.3 val_worst_mae=0.0151 | best=0.1083
[da_gps chunk_parent addons] epoch 100 | territory_frac=0.1795 | train plain_ce=0.8996 hybrid_total=0.9539 ordinal_cost=0.0543 ordinal_frac=6.0% reg_tap_mae_pu=0.0029 reg_tap_acc=0.6124 | volt loss_w=0.0132 loss_u=0.0133 delta=-0.0001 | val_reg=0.8896 val_tap_mae=0.0029 val_tap_acc=0.6177 val_volt_uniform=0.0123 val_hybrid_total=0.9423 val_plain_ce=0.8896 val_ordinal_cost=0.0527 val_ordinal_frac=5.9% | val_tot=0.1085 [1*volt(0.0123)+0.1*cap(0.0665)+0.1*reg(0.8894)+0.1*meta(0.0068)]
[da_gps addons counterfactual] epoch 100 | territory: val_reg=0.8896 no_territory=3.3794 delta=-2.4898 (negative helps) | hybrid: val_plain_ce=0.8896 val_ordinal_cost=0.0527 val_hybrid_total=0.9423 delta_hybrid_vs_plain=0.0527 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6177 | voltage: val_uniform=0.0123 val_weighted=0.0121 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  110/200 | train_tot=0.1150 train_volt=0.0128 train_cap=0.0660 train_reg=0.9478 train_meta_aux=0.0074 val_meta_aux=0.0076 | val_tot=0.1098 val_volt=0.0117 val_cap=0.0779 val_reg=0.8952 | val_r2_mean=0.9303 val_r2_min=0.1007 val_r2_min_node=190-8593.3 val_worst_mae=0.0148 | best=0.1083
[da_gps chunk_parent addons] epoch 110 | territory_frac=0.1795 | train plain_ce=0.8936 hybrid_total=0.9474 ordinal_cost=0.0538 ordinal_frac=6.0% reg_tap_mae_pu=0.0029 reg_tap_acc=0.6140 | volt loss_w=0.0128 loss_u=0.0129 delta=-0.0001 | val_reg=0.8953 val_tap_mae=0.0029 val_tap_acc=0.6117 val_volt_uniform=0.0117 val_hybrid_total=0.9485 val_plain_ce=0.8953 val_ordinal_cost=0.0532 val_ordinal_frac=5.9% | val_tot=0.1098 [1*volt(0.0117)+0.1*cap(0.0779)+0.1*reg(0.8952)+0.1*meta(0.0076)]
[da_gps addons counterfactual] epoch 110 | territory: val_reg=0.8953 no_territory=3.3887 delta=-2.4934 (negative helps) | hybrid: val_plain_ce=0.8953 val_ordinal_cost=0.0532 val_hybrid_total=0.9485 delta_hybrid_vs_plain=0.0532 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6117 | voltage: val_uniform=0.0117 val_weighted=0.0116 viol_frac=0.2970 delta=-0.0001
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  120/200 | train_tot=0.1143 train_volt=0.0126 train_cap=0.0654 train_reg=0.9444 train_meta_aux=0.0073 val_meta_aux=0.0058 | val_tot=0.1095 val_volt=0.0127 val_cap=0.0688 val_reg=0.8925 | val_r2_mean=0.9287 val_r2_min=0.0561 val_r2_min_node=190-8593.3 val_worst_mae=0.0149 | best=0.1083
[da_gps chunk_parent addons] epoch 120 | territory_frac=0.1795 | train plain_ce=0.8903 hybrid_total=0.9439 ordinal_cost=0.0536 ordinal_frac=6.0% reg_tap_mae_pu=0.0029 reg_tap_acc=0.6156 | volt loss_w=0.0126 loss_u=0.0127 delta=-0.0001 | val_reg=0.8927 val_tap_mae=0.0029 val_tap_acc=0.6138 val_volt_uniform=0.0128 val_hybrid_total=0.9457 val_plain_ce=0.8927 val_ordinal_cost=0.0530 val_ordinal_frac=5.9% | val_tot=0.1095 [1*volt(0.0127)+0.1*cap(0.0688)+0.1*reg(0.8925)+0.1*meta(0.0058)]
[da_gps addons counterfactual] epoch 120 | territory: val_reg=0.8927 no_territory=3.2343 delta=-2.3416 (negative helps) | hybrid: val_plain_ce=0.8927 val_ordinal_cost=0.0530 val_hybrid_total=0.9457 delta_hybrid_vs_plain=0.0530 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6138 | voltage: val_uniform=0.0128 val_weighted=0.0128 viol_frac=0.2970 delta=0.0001
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  130/200 | train_tot=0.1137 train_volt=0.0125 train_cap=0.0644 train_reg=0.9407 train_meta_aux=0.0072 val_meta_aux=0.0065 | val_tot=0.1110 val_volt=0.0128 val_cap=0.0641 val_reg=0.9117 | val_r2_mean=0.9294 val_r2_min=0.0994 val_r2_min_node=190-8593.3 val_worst_mae=0.0148 | best=0.1083
[da_gps chunk_parent addons] epoch 130 | territory_frac=0.1795 | train plain_ce=0.8873 hybrid_total=0.9406 ordinal_cost=0.0533 ordinal_frac=6.0% reg_tap_mae_pu=0.0029 reg_tap_acc=0.6162 | volt loss_w=0.0125 loss_u=0.0126 delta=-0.0001 | val_reg=0.9118 val_tap_mae=0.0030 val_tap_acc=0.6062 val_volt_uniform=0.0128 val_hybrid_total=0.9656 val_plain_ce=0.9118 val_ordinal_cost=0.0539 val_ordinal_frac=5.9% | val_tot=0.1110 [1*volt(0.0128)+0.1*cap(0.0641)+0.1*reg(0.9117)+0.1*meta(0.0065)]
[da_gps addons counterfactual] epoch 130 | territory: val_reg=0.9118 no_territory=3.3340 delta=-2.4222 (negative helps) | hybrid: val_plain_ce=0.9118 val_ordinal_cost=0.0539 val_hybrid_total=0.9656 delta_hybrid_vs_plain=0.0539 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6062 | voltage: val_uniform=0.0128 val_weighted=0.0126 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  140/200 | train_tot=0.1130 train_volt=0.0123 train_cap=0.0640 train_reg=0.9360 train_meta_aux=0.0072 val_meta_aux=0.0056 | val_tot=0.1068 val_volt=0.0120 val_cap=0.0663 val_reg=0.8756 | val_r2_mean=0.9318 val_r2_min=0.0903 val_r2_min_node=190-8593.3 val_worst_mae=0.0143 | best=0.1068
[da_gps chunk_parent addons] epoch 140 | territory_frac=0.1795 | train plain_ce=0.8828 hybrid_total=0.9358 ordinal_cost=0.0530 ordinal_frac=6.0% reg_tap_mae_pu=0.0029 reg_tap_acc=0.6192 | volt loss_w=0.0123 loss_u=0.0124 delta=-0.0001 | val_reg=0.8757 val_tap_mae=0.0028 val_tap_acc=0.6229 val_volt_uniform=0.0120 val_hybrid_total=0.9273 val_plain_ce=0.8757 val_ordinal_cost=0.0516 val_ordinal_frac=5.9% | val_tot=0.1068 [1*volt(0.0120)+0.1*cap(0.0663)+0.1*reg(0.8756)+0.1*meta(0.0056)]
[da_gps addons counterfactual] epoch 140 | territory: val_reg=0.8757 no_territory=3.1336 delta=-2.2579 (negative helps) | hybrid: val_plain_ce=0.8757 val_ordinal_cost=0.0516 val_hybrid_total=0.9273 delta_hybrid_vs_plain=0.0516 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6229 | voltage: val_uniform=0.0120 val_weighted=0.0119 viol_frac=0.2970 delta=-0.0001
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  150/200 | train_tot=0.1126 train_volt=0.0122 train_cap=0.0638 train_reg=0.9338 train_meta_aux=0.0070 val_meta_aux=0.0068 | val_tot=0.1067 val_volt=0.0108 val_cap=0.0680 val_reg=0.8837 | val_r2_mean=0.9320 val_r2_min=0.1124 val_r2_min_node=190-8593.3 val_worst_mae=0.0139 | best=0.1067
[da_gps chunk_parent addons] epoch 150 | territory_frac=0.1795 | train plain_ce=0.8804 hybrid_total=0.9332 ordinal_cost=0.0528 ordinal_frac=6.0% reg_tap_mae_pu=0.0029 reg_tap_acc=0.6190 | volt loss_w=0.0122 loss_u=0.0123 delta=-0.0001 | val_reg=0.8839 val_tap_mae=0.0029 val_tap_acc=0.6193 val_volt_uniform=0.0109 val_hybrid_total=0.9358 val_plain_ce=0.8839 val_ordinal_cost=0.0520 val_ordinal_frac=5.9% | val_tot=0.1067 [1*volt(0.0108)+0.1*cap(0.0680)+0.1*reg(0.8837)+0.1*meta(0.0068)]
[da_gps addons counterfactual] epoch 150 | territory: val_reg=0.8839 no_territory=3.3245 delta=-2.4407 (negative helps) | hybrid: val_plain_ce=0.8839 val_ordinal_cost=0.0520 val_hybrid_total=0.9358 delta_hybrid_vs_plain=0.0520 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6193 | voltage: val_uniform=0.0109 val_weighted=0.0106 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  160/200 | train_tot=0.1121 train_volt=0.0120 train_cap=0.0634 train_reg=0.9300 train_meta_aux=0.0070 val_meta_aux=0.0063 | val_tot=0.1038 val_volt=0.0109 val_cap=0.0631 val_reg=0.8595 | val_r2_mean=0.9321 val_r2_min=0.0360 val_r2_min_node=190-8593.3 val_worst_mae=0.0139 | best=0.1038
[da_gps chunk_parent addons] epoch 160 | territory_frac=0.1795 | train plain_ce=0.8769 hybrid_total=0.9293 ordinal_cost=0.0525 ordinal_frac=6.0% reg_tap_mae_pu=0.0028 reg_tap_acc=0.6208 | volt loss_w=0.0120 loss_u=0.0121 delta=-0.0001 | val_reg=0.8596 val_tap_mae=0.0028 val_tap_acc=0.6282 val_volt_uniform=0.0109 val_hybrid_total=0.9110 val_plain_ce=0.8596 val_ordinal_cost=0.0514 val_ordinal_frac=6.0% | val_tot=0.1038 [1*volt(0.0109)+0.1*cap(0.0631)+0.1*reg(0.8595)+0.1*meta(0.0063)]
[da_gps addons counterfactual] epoch 160 | territory: val_reg=0.8596 no_territory=3.1827 delta=-2.3231 (negative helps) | hybrid: val_plain_ce=0.8596 val_ordinal_cost=0.0514 val_hybrid_total=0.9110 delta_hybrid_vs_plain=0.0514 (positive hurts val_reg if hybrid used) ordinal_frac=6.0% val_tap_acc=0.6282 | voltage: val_uniform=0.0109 val_weighted=0.0108 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  170/200 | train_tot=0.1118 train_volt=0.0120 train_cap=0.0630 train_reg=0.9287 train_meta_aux=0.0070 val_meta_aux=0.0055 | val_tot=0.1040 val_volt=0.0108 val_cap=0.0643 val_reg=0.8622 | val_r2_mean=0.9332 val_r2_min=0.1056 val_r2_min_node=190-8593.3 val_worst_mae=0.0138 | best=0.1038
[da_gps chunk_parent addons] epoch 170 | territory_frac=0.1795 | train plain_ce=0.8760 hybrid_total=0.9283 ordinal_cost=0.0524 ordinal_frac=6.0% reg_tap_mae_pu=0.0028 reg_tap_acc=0.6215 | volt loss_w=0.0120 loss_u=0.0121 delta=-0.0001 | val_reg=0.8623 val_tap_mae=0.0028 val_tap_acc=0.6273 val_volt_uniform=0.0108 val_hybrid_total=0.9131 val_plain_ce=0.8623 val_ordinal_cost=0.0508 val_ordinal_frac=5.9% | val_tot=0.1040 [1*volt(0.0108)+0.1*cap(0.0643)+0.1*reg(0.8622)+0.1*meta(0.0055)]
[da_gps addons counterfactual] epoch 170 | territory: val_reg=0.8623 no_territory=3.2513 delta=-2.3890 (negative helps) | hybrid: val_plain_ce=0.8623 val_ordinal_cost=0.0508 val_hybrid_total=0.9131 delta_hybrid_vs_plain=0.0508 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6273 | voltage: val_uniform=0.0108 val_weighted=0.0106 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  180/200 | train_tot=0.1115 train_volt=0.0118 train_cap=0.0630 train_reg=0.9272 train_meta_aux=0.0068 val_meta_aux=0.0055 | val_tot=0.1045 val_volt=0.0114 val_cap=0.0651 val_reg=0.8603 | val_r2_mean=0.9322 val_r2_min=0.1397 val_r2_min_node=190-8593.3 val_worst_mae=0.0137 | best=0.1038
[da_gps chunk_parent addons] epoch 180 | territory_frac=0.1795 | train plain_ce=0.8745 hybrid_total=0.9268 ordinal_cost=0.0523 ordinal_frac=6.0% reg_tap_mae_pu=0.0028 reg_tap_acc=0.6217 | volt loss_w=0.0118 loss_u=0.0119 delta=-0.0001 | val_reg=0.8605 val_tap_mae=0.0028 val_tap_acc=0.6281 val_volt_uniform=0.0114 val_hybrid_total=0.9113 val_plain_ce=0.8605 val_ordinal_cost=0.0509 val_ordinal_frac=5.9% | val_tot=0.1045 [1*volt(0.0114)+0.1*cap(0.0651)+0.1*reg(0.8603)+0.1*meta(0.0055)]
[da_gps addons counterfactual] epoch 180 | territory: val_reg=0.8605 no_territory=3.1573 delta=-2.2968 (negative helps) | hybrid: val_plain_ce=0.8605 val_ordinal_cost=0.0509 val_hybrid_total=0.9113 delta_hybrid_vs_plain=0.0509 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6281 | voltage: val_uniform=0.0114 val_weighted=0.0113 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  190/200 | train_tot=0.1109 train_volt=0.0116 train_cap=0.0625 train_reg=0.9231 train_meta_aux=0.0067 val_meta_aux=0.0054 | val_tot=0.1036 val_volt=0.0110 val_cap=0.0625 val_reg=0.8584 | val_r2_mean=0.9333 val_r2_min=0.1505 val_r2_min_node=190-8593.3 val_worst_mae=0.0136 | best=0.1036
[da_gps chunk_parent addons] epoch 190 | territory_frac=0.1795 | train plain_ce=0.8707 hybrid_total=0.9227 ordinal_cost=0.0520 ordinal_frac=6.0% reg_tap_mae_pu=0.0028 reg_tap_acc=0.6234 | volt loss_w=0.0116 loss_u=0.0118 delta=-0.0001 | val_reg=0.8586 val_tap_mae=0.0028 val_tap_acc=0.6293 val_volt_uniform=0.0110 val_hybrid_total=0.9094 val_plain_ce=0.8586 val_ordinal_cost=0.0507 val_ordinal_frac=5.9% | val_tot=0.1036 [1*volt(0.0110)+0.1*cap(0.0625)+0.1*reg(0.8584)+0.1*meta(0.0054)]
[da_gps addons counterfactual] epoch 190 | territory: val_reg=0.8586 no_territory=3.0804 delta=-2.2218 (negative helps) | hybrid: val_plain_ce=0.8586 val_ordinal_cost=0.0507 val_hybrid_total=0.9094 delta_hybrid_vs_plain=0.0507 (positive hurts val_reg if hybrid used) ordinal_frac=5.9% val_tap_acc=0.6293 | voltage: val_uniform=0.0110 val_weighted=0.0108 viol_frac=0.2970 delta=-0.0002
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
[da_gps chunk_parent] epoch  200/200 | train_tot=0.1108 train_volt=0.0117 train_cap=0.0624 train_reg=0.9217 train_meta_aux=0.0067 val_meta_aux=0.0048 | val_tot=0.1048 val_volt=0.0108 val_cap=0.0637 val_reg=0.8709 | val_r2_mean=0.9321 val_r2_min=0.1216 val_r2_min_node=190-8593.3 val_worst_mae=0.0136 | best=0.1036
[da_gps chunk_parent addons] epoch 200 | territory_frac=0.1795 | train plain_ce=0.8691 hybrid_total=0.9210 ordinal_cost=0.0519 ordinal_frac=6.0% reg_tap_mae_pu=0.0028 reg_tap_acc=0.6240 | volt loss_w=0.0117 loss_u=0.0118 delta=-0.0001 | val_reg=0.8711 val_tap_mae=0.0028 val_tap_acc=0.6233 val_volt_uniform=0.0109 val_hybrid_total=0.9219 val_plain_ce=0.8711 val_ordinal_cost=0.0508 val_ordinal_frac=5.8% | val_tot=0.1048 [1*volt(0.0108)+0.1*cap(0.0637)+0.1*reg(0.8709)+0.1*meta(0.0048)]
[da_gps addons counterfactual] epoch 200 | territory: val_reg=0.8711 no_territory=3.2288 delta=-2.3578 (negative helps) | hybrid: val_plain_ce=0.8711 val_ordinal_cost=0.0508 val_hybrid_total=0.9219 delta_hybrid_vs_plain=0.0508 (positive hurts val_reg if hybrid used) ordinal_frac=5.8% val_tap_acc=0.6233 | voltage: val_uniform=0.0109 val_weighted=0.0108 viol_frac=0.2970 delta=-0.0000
  periodic checkpoint -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4692: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5286: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4692: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5286: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
Val |V| MAE=0.003950  angle MAE=0.220471  Re/Im MSE(nrm)=0.010981
Test |V| MAE=0.003940  angle MAE=0.222060  Re/Im MSE(nrm)=0.010978  cap_BCE=0.061081  reg_CE=0.854685  reg_acc=0.6266  meta_aux_MSE(nrm)=0.005578  meta_aux_MSE(raw)=2696.179855  time=31932.8s
[da_gps chunk_parent] Test per-head cap_BCE:
  cap_capbank0a_n_steps_on=0.015539
  cap_capbank0b_n_steps_on=0.018422
  cap_capbank0c_n_steps_on=0.044341
  cap_capbank1a_n_steps_on=0.032398
  cap_capbank1b_n_steps_on=0.044959
  cap_capbank1c_n_steps_on=0.089490
  cap_capbank2a_n_steps_on=0.087334
  cap_capbank2b_n_steps_on=0.108343
  cap_capbank2c_n_steps_on=0.169985
  cap_capbank3_n_steps_on=0.000000
[da_gps chunk_parent] Test per-head reg_MSE / reg_MAE (nrm / tap pu):
  reg_feeder_rega_tap_pu: MSE nrm=0.275416 pu=0.275416  MAE nrm=0.259836 pu=0.259836
  reg_feeder_regb_tap_pu: MSE nrm=0.250858 pu=0.250858  MAE nrm=0.245577 pu=0.245577
  reg_feeder_regc_tap_pu: MSE nrm=0.249010 pu=0.249010  MAE nrm=0.237391 pu=0.237391
  reg_vreg2_a_tap_pu: MSE nrm=0.776472 pu=0.776472  MAE nrm=0.489966 pu=0.489966
  reg_vreg2_b_tap_pu: MSE nrm=0.923158 pu=0.923158  MAE nrm=0.583047 pu=0.583047
  reg_vreg2_c_tap_pu: MSE nrm=1.144574 pu=1.144574  MAE nrm=0.680354 pu=0.680354
  reg_vreg3_a_tap_pu: MSE nrm=0.477555 pu=0.477555  MAE nrm=0.322287 pu=0.322287
  reg_vreg3_b_tap_pu: MSE nrm=0.804595 pu=0.804595  MAE nrm=0.511487 pu=0.511487
  reg_vreg3_c_tap_pu: MSE nrm=1.094666 pu=1.094666  MAE nrm=0.633087 pu=0.633087
  reg_vreg4_a_tap_pu: MSE nrm=0.465936 pu=0.465936  MAE nrm=0.376683 pu=0.376683
  reg_vreg4_b_tap_pu: MSE nrm=0.556773 pu=0.556773  MAE nrm=0.440586 pu=0.440586
  reg_vreg4_c_tap_pu: MSE nrm=0.736995 pu=0.736995  MAE nrm=0.550040 pu=0.550040
[da_gps chunk_parent] Test per-head meta_aux_MSE (nrm / raw):
  pv_pv2_p_post_kw: nrm=0.003192  raw=525.122205
  pv_pv2_q_post_kvar: nrm=0.017759  raw=361.321142
  p_loss_total_post_kw: nrm=0.000778  raw=1955.726455
  q_loss_total_post_kvar: nrm=0.000581  raw=7942.549801
Saved /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/da_gps_multitask_best.pt

Training completed.
Run dir: /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401
Checkpoint (best): /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/da_gps_multitask_best.pt
Checkpoint (last): /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/training_last.pt
Regulator classes: /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/reg_class_tables.json
Report: /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401/da_gps_report.json

=== Section 8 prerequisite ===
Baseline run folder:  /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401
Section 8 expects:    /content/drive/MyDrive/datasets_gnn2/runs/da_gps_chunked_l2_h64_mvagg_gine_metaaux_regce_addons_moderate_full_20260708_050401
Checkpoint for init:  da_gps_multitask_best.pt (or training_last.pt)
WARNING: run dir is NOT under MyDrive/datasets_gnn2 — copy to Drive before disconnecting the VM or Section 8 will fail.
Smoke compare manifest (baseline) -> /content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/_smoke_compare_manifest.json

## Territory-only ablation: β=7.0; no hybrid ordinal; no volt violation weight

Self-contained add-on run: keeps `--attn_reg_territory_bias` with β=7.0; disables hybrid ordinal tap loss and voltage violation weighting.


In [ ]:
import os
import sys
import subprocess
import datetime
import warnings
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"
COLAB_CHUNK_DEFAULT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
COLAB_NODE_PE_DEFAULT = (
    COLAB_CHUNK_DEFAULT
    / "run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv"
)
COLAB_RUNS_PARENT = Path("/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints")
HOP_CSV_NAME = "load_hop_distance_to_each_regulator_all_index_nodes.csv"
WIN_CHUNK_DEFAULT = Path(r"D:\datasets\original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")


def _is_windows_drive_path(path) -> bool:
    s = str(path).strip().replace("/", "\\")
    return len(s) >= 2 and s[1] == ":" and s[0].isalpha()


def _on_colab() -> bool:
    return Path("/content").is_dir()


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _resolve_data_path(path: Path, *, label: str, colab_fallback: Path | None = None) -> Path:
    """Resolve a data path; never join Windows drive letters with cwd on Linux."""
    raw = str(path)
    if _is_windows_drive_path(raw):
        if _on_colab() and colab_fallback is not None:
            warnings.warn(
                f"{label}={raw!r} is a Windows path on Linux/Colab; using {colab_fallback} instead.",
                UserWarning,
                stacklevel=2,
            )
            return colab_fallback.expanduser().resolve()
        if _on_colab():
            raise ValueError(
                f"{label}={raw!r} is a Windows absolute path and invalid on Colab. "
                f"Mount Drive and use e.g. {COLAB_CHUNK_DEFAULT}"
            )
        return Path(raw).expanduser().resolve()
    p = Path(path).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (Path.cwd() / p).resolve()


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(
            f"No run_*/gnn_node_index_master.csv under {chunk_parent}"
        )
    return hits[0]


def _resolve_hop_csv(
    *,
    repo: Path,
    chunk_parent: Path,
    mydrive_data: Path | None,
) -> Path | None:
    """Resolve regulator hop CSV for --attn_reg_territory_bias (first existing candidate)."""
    env_raw = os.environ.get("GNN2_HOP_CSV", "").strip()
    candidates: list[Path] = []
    if env_raw:
        p = Path(env_raw).expanduser()
        candidates.append(p.resolve() if p.is_absolute() else (repo / p).resolve())
    if mydrive_data is not None:
        candidates.append((mydrive_data / HOP_CSV_NAME).resolve())
    cp = chunk_parent.resolve()
    candidates.extend(
        [
            (cp / ".." / HOP_CSV_NAME).resolve(),
            (cp.parent / HOP_CSV_NAME).resolve(),
        ]
    )
    candidates.extend(
        [
            (repo / "datasets_gnn2_from pc" / HOP_CSV_NAME).resolve(),
            (repo / "datasets_gnn2" / HOP_CSV_NAME).resolve(),
        ]
    )
    seen: set[Path] = set()
    for c in candidates:
        if c in seen:
            continue
        seen.add(c)
        if c.is_file():
            return c
    return None

def _sorted_run_chunk_dirs(chunk_parent: Path) -> list[Path]:
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )


def _smoke_chunk_subdir_glob(chunk_parent: Path, count: int) -> str:
    """First `count` run_* names, comma-separated for --chunk_subdir_glob."""
    names = [p.name for p in _sorted_run_chunk_dirs(chunk_parent)]
    if len(names) < count:
        raise ValueError(
            f"SMOKE_CHUNK_COUNT={count} but only {len(names)} run_* under {chunk_parent}"
        )
    return ",".join(names[:count])


def _chunks_from_subdir_glob(chunk_parent: Path, glob_pat: str) -> list[Path]:
    import fnmatch

    glob_pat = str(glob_pat).strip()
    if "," in glob_pat:
        allowed = {s.strip() for s in glob_pat.split(",") if s.strip()}
        chunks = sorted(
            (p for p in chunk_parent.iterdir() if p.is_dir() and p.name in allowed),
            key=lambda p: p.name,
        )
        missing = allowed - {p.name for p in chunks}
        if missing:
            raise FileNotFoundError(f"Missing smoke chunk folders: {sorted(missing)}")
        return chunks
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and fnmatch.fnmatch(p.name, glob_pat)),
        key=lambda p: p.name,
    )


def _write_smoke_compare_manifest(
    runs_parent: Path,
    *,
    out_dir: Path,
    physics_weight: float,
    chunk_glob: str,
    n_chunks: int,
    epochs: int,
    seed: int,
) -> None:
    import json as _json

    path = runs_parent / "_smoke_compare_manifest.json"
    manifest = {}
    if path.is_file():
        manifest = _json.loads(path.read_text(encoding="utf-8"))
    key = "physics" if physics_weight > 0 else "baseline"
    manifest[key] = {
        "run_dir": str(out_dir.resolve()),
        "physics_weight": physics_weight,
        "chunk_glob": chunk_glob,
        "n_chunks": n_chunks,
        "epochs": epochs,
        "seed": seed,
    }
    path.write_text(_json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Smoke compare manifest ({key}) -> {path}")


# Colab: mount Drive when chunks/caches live under MyDrive
if _on_colab() and not _drive_mounted():
    from google.colab import drive
    drive.mount("/content/drive")

# --- FULL training (set True only for quick smoke/debug) ---
SMOKE_TEST = False
SMOKE_CHUNK_COUNT = 3  # used only when SMOKE_TEST=True
SMOKE_EPOCHS = 15
SMOKE_PATIENCE = 5
SMOKE_SEED = 42
FULL_EPOCHS = 200
FULL_PATIENCE = 30
TRAIN_SEED = 42

# TERRITORY-ONLY ablation (2026-07): keep territory attention bias; disable hybrid ordinal + volt violation weight.
ATTN_REG_TERRITORY_BETA = 7.0   # was 6.0 tuned / 9.0 extreme / 2.0 default
_DA_CACHE_NAME = "da_gps_chunked_mvagg_smoke_gine" if SMOKE_TEST else "da_gps_chunked_mvagg_full_gine"

# --- paths: auto-detect Colab + Drive; override below if needed ---
_env_repo = os.environ.get("GNN2_REPO_ROOT", "").strip()
REPO = Path(_env_repo).expanduser().resolve() if _env_repo else Path.cwd().resolve()
if not (REPO / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
    raise FileNotFoundError(f"Repo root missing trainer script: {REPO}")

if _on_colab():
    if not _drive_mounted():
        raise RuntimeError(
            "Colab requires Google Drive mounted. Re-run this cell and approve Drive access, "
            "or: from google.colab import drive; drive.mount('/content/drive')"
        )
    CHUNK_PARENT = COLAB_CHUNK_DEFAULT
    NODE_PE_CSV = COLAB_NODE_PE_DEFAULT
    # Caches must live on Drive to reuse across Colab sessions (/content is wiped on reset)
    DA_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = COLAB_RUNS_PARENT
elif os.name == "nt":
    CHUNK_PARENT = WIN_CHUNK_DEFAULT
    NODE_PE_CSV = None  # auto: first run_*/gnn_node_index_master.csv under CHUNK_PARENT
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"
else:
    CHUNK_PARENT = REPO / "datasets_gnn2_from pc/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
    NODE_PE_CSV = None
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"

# Optional overrides (use POSIX paths on Colab, not D:\...):
# CHUNK_PARENT = Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")
# NODE_PE_CSV = CHUNK_PARENT / "run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv"
# RUNS_PARENT = MYDRIVE_DATA / "runs"  # persist checkpoints on Drive instead of ephemeral clone

# --- lighter architecture (locked in for add-ons full run) ---
LIGHTER_HIDDEN = 64
LIGHTER_LAYERS = 2
LIGHTER_HEADS = 2
LIGHTER_NODE_EMB_DIM = 2
LIGHTER_EDGE_EMB_DIM = 0

PHYSICS_WEIGHT = 0.0  # baseline; early_stop_on=total

# Explicit PF balance nodes when physics on (1177 hetero MV load nodes; chunk-safe)
PF_BALANCE_NODE_LIST_CSV = REPO / "colab_pf_data/pf_balance_nodes_explicit.csv"

META_AUX_COLS = "pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar"
NUM_WORKERS = 0 if os.name == "nt" else 4

os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

_colab_fb = COLAB_CHUNK_DEFAULT if _on_colab() else None
chunk_parent = _resolve_data_path(CHUNK_PARENT, label="CHUNK_PARENT", colab_fallback=_colab_fb)
if SMOKE_TEST:
    CHUNK_GLOB = _smoke_chunk_subdir_glob(chunk_parent, SMOKE_CHUNK_COUNT)
    EPOCHS = SMOKE_EPOCHS
    PATIENCE = SMOKE_PATIENCE
    SEED = SMOKE_SEED
else:
    CHUNK_GLOB = "run_*"
    EPOCHS = FULL_EPOCHS
    PATIENCE = FULL_PATIENCE
    SEED = TRAIN_SEED
da_cache_root = _resolve_data_path(
    DA_CACHE_ROOT,
    label="DA_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}" if _on_colab() else None,
)
gnn_cache_root = _resolve_data_path(
    GNN_CACHE_ROOT,
    label="GNN_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine" if _on_colab() else None,
)
runs_parent = _resolve_data_path(
    RUNS_PARENT,
    label="RUNS_PARENT",
    colab_fallback=MYDRIVE_DATA / "runs" if _on_colab() else None,
)

# --- PF topology root (repo colab_pf_data/ after git pull, or Drive dailyagg fallback) ---
from gnn2_pf_data_paths import PF_CAP_NODES_REL, PF_REG_CATALOG_REL, resolve_pf_catalog_paths

PF_DATA_ROOT = None
if PHYSICS_WEIGHT > 0:
    _reg_cat, _cap_map, PF_DATA_ROOT = resolve_pf_catalog_paths(
        repo=REPO,
        preferred_root=None,
        chunk_parent=chunk_parent,
    )

print("=== Preflight (add-ons TERRITORY-ONLY experiment) ===")
print(f"ADDON COEFFS:   territory_beta={ATTN_REG_TERRITORY_BETA} (no hybrid ordinal; no volt violation weight)")
print(f"REPO:           {REPO}")
print(f"SMOKE_TEST:     {SMOKE_TEST}")
print(f"SMOKE_CHUNK_COUNT: {SMOKE_CHUNK_COUNT}")
print(f"CHUNK_GLOB:     {CHUNK_GLOB}")
print(f"EPOCHS:         {EPOCHS}")
print(f"PATIENCE:       {PATIENCE}")
print(f"SEED:           {SEED}")
print(f"CHUNK_PARENT:   {chunk_parent}")
if not chunk_parent.is_dir():
    raise FileNotFoundError(f"CHUNK_PARENT not found: {chunk_parent}")

run_preview = sorted(
    p.name for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")
)[:5]
print(f"run_* preview:  {run_preview}")

if NODE_PE_CSV is None:
    node_pe = _find_node_pe_csv(chunk_parent)
else:
    node_pe = _resolve_data_path(NODE_PE_CSV, label="NODE_PE_CSV", colab_fallback=COLAB_NODE_PE_DEFAULT if _on_colab() else None)
    if not node_pe.is_file():
        warnings.warn(
            f"NODE_PE_CSV not found at {node_pe}; auto-discovering under CHUNK_PARENT.",
            UserWarning,
            stacklevel=2,
        )
        node_pe = _find_node_pe_csv(chunk_parent)

print(f"NODE_PE_CSV:    {node_pe}")
print(f"DA_CACHE_ROOT:  {da_cache_root}")
print(f"GNN_CACHE_ROOT: {gnn_cache_root}")
print(f"RUNS_PARENT:    {runs_parent}")

_mydrive_for_hop = MYDRIVE_DATA if _on_colab() and _drive_mounted() else None
hop_csv = _resolve_hop_csv(repo=REPO, chunk_parent=chunk_parent, mydrive_data=_mydrive_for_hop)
_hop_status = "OK" if hop_csv is not None else "MISSING"
print(f"HOP_CSV:        {_hop_status}" + (f"  {hop_csv}" if hop_csv else ""))
if hop_csv is None:
    _expected = MYDRIVE_DATA / HOP_CSV_NAME if _on_colab() else REPO / "datasets_gnn2_from pc" / HOP_CSV_NAME
    raise FileNotFoundError(
        "Regulator hop CSV required for --attn_reg_territory_bias but not found. "
        f"Generate with compute_hop_distance_all_index_nodes.py or copy to {_expected}"
    )
os.environ["GNN2_HOP_CSV"] = str(hop_csv)

from da_gps_hop_attention_ratios import TARGET_REG_COLS, validate_reg_hop_csv

_reg_csv = None
for _cand in (
    chunk_parent.parent / "regulator_involved_nodes.csv",
    REPO / "datasets_gnn2_from pc" / "loadtype_8500_dailyagg" / "regulator_involved_nodes.csv",
):
    if _cand.is_file():
        _reg_csv = _cand
        break
_hop_map = validate_reg_hop_csv(
    hop_csv,
    list(TARGET_REG_COLS),
    regulator_csv=_reg_csv,
)
print("HOP preflight (phase-consistent):", ", ".join(f"{k}->{v}" for k, v in _hop_map.items()))

# Phase-consistent hop column mapping preflight (territory bias is harmful if wrong).
import pandas as pd
from da_gps_hop_attention_ratios import REG_COL_TO_HOP_COL, validate_reg_hop_csv
from train_da_gps_multitask_complex_voltage import TARGET_REG_COLS

_hop_node_names = pd.read_csv(node_pe)["node"].astype(str).tolist()
_reg_csv_candidates = [
    chunk_parent.parent / "regulator_involved_nodes.csv",
    REPO / "datasets_gnn2_from pc" / "loadtype_8500_dailyagg" / "regulator_involved_nodes.csv",
]
_hop_reg_csv = next((p for p in _reg_csv_candidates if p.is_file()), None)
_hop_pairs = validate_reg_hop_csv(
    hop_csv,
    list(TARGET_REG_COLS),
    node_names=_hop_node_names,
    reg_col_to_hop_col=REG_COL_TO_HOP_COL,
    regulator_csv=_hop_reg_csv,
)
print("HOP mapping preflight (reg_col -> hop_col):")
for _rc, _hc in _hop_pairs.items():
    print(f"  {_rc} -> {_hc}")

if PHYSICS_WEIGHT > 0:
    assert PF_DATA_ROOT is not None
    _pf_checks = [
        ("reg_catalog", PF_DATA_ROOT / PF_REG_CATALOG_REL),
        ("cap_nodes", PF_DATA_ROOT / PF_CAP_NODES_REL),
        ("electrical_distance", PF_DATA_ROOT / "electrical_distance_from_substation.csv"),
        (
            "hetero_mv_nodes",
            PF_DATA_ROOT / "Heterogenous GNN dataset/nodes/hetero_mv_nodes_load_transformer.csv",
        ),
        ("bus_kv_cache", PF_DATA_ROOT / "bus_kv_base_by_node.csv"),
    ]
    print(f"PF_DATA_ROOT:   {PF_DATA_ROOT}")
    for label, p in _pf_checks:
        status = "OK" if p.is_file() else "MISSING"
        print(f"  PF {label}: {status}  {p}")
        if not p.is_file():
            raise FileNotFoundError(f"Physics preflight missing {label}: {p}")
else:
    print("PF_DATA_ROOT:   (skipped — PHYSICS_WEIGHT=0)")

print("=================")

da_cache_root.mkdir(parents=True, exist_ok=True)
gnn_cache_root.mkdir(parents=True, exist_ok=True)
runs_parent.mkdir(parents=True, exist_ok=True)

chunks = _chunks_from_subdir_glob(chunk_parent, CHUNK_GLOB)
if not chunks:
    raise RuntimeError(f"No folders for CHUNK_GLOB={CHUNK_GLOB!r} under {chunk_parent}")

for p in chunks:
    for name in (
        "gnn_node_features_and_targets_mvagg.csv",
        "gnn_edges_phase_static.csv",
        "gnn_sample_meta.csv",
    ):
        if not (p / name).is_file():
            raise FileNotFoundError(f"Missing {name} in {p.name}")

print(f"Found {len(chunks)} chunk(s) for CHUNK_GLOB under {chunk_parent}")

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_pf_suffix = "_pf" if PHYSICS_WEIGHT > 0 else ""
_addon_run_suffix = "_addons_territory_only_smoke" if SMOKE_TEST else "_addons_territory_only_full"
out_dir = runs_parent / (
    f"da_gps_chunked_l{LIGHTER_LAYERS}_h{LIGHTER_HIDDEN}_mvagg_gine_metaaux_regce{_pf_suffix}{_addon_run_suffix}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

_early_stop = "voltage" if PHYSICS_WEIGHT > 0 else "total"

cmd = [
    sys.executable, "-u", "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent", str(chunk_parent),
    "--chunk_subdir_glob", CHUNK_GLOB,
    "--nodes_csv", "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv", "gnn_edges_phase_static.csv",
    "--meta_csv", "gnn_sample_meta.csv",
    "--node_feature_cols", "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv", str(node_pe),
    "--node_pe_cols", "auto",
    "--n_system_tokens", "10",
    "--aux_meta_cols", META_AUX_COLS,
    "--lambda_pv", "0.1",
    "--out_dir", str(out_dir),
    "--cache_dir", str(da_cache_root),
    "--bootstrap_gnn_cache_dir", str(gnn_cache_root),
    "--epochs", str(EPOCHS),
    "--batch_size", "64",
    "--hidden", str(LIGHTER_HIDDEN),
    "--layers", str(LIGHTER_LAYERS),
    "--heads", str(LIGHTER_HEADS),
    "--node_emb_dim", str(LIGHTER_NODE_EMB_DIM),
    "--edge_emb_dim", str(LIGHTER_EDGE_EMB_DIM),
    "--lr", "5e-4",
    "--weight_decay", "1e-5",
    "--lambda_cap", "0.1",
    "--lambda_reg", "0.1",
    "--reg_loss", "ce",
    "--per_device_cap_head",
    "--per_device_reg_head",
    "--patience", str(PATIENCE),
    "--seed", str(SEED),
    "--train_frac", "0.80",
    "--val_frac", "0.10",
    "--sample_frac", "1.0",
    "--num_workers", str(NUM_WORKERS),
    "--eval_every", "10",
    "--log_every", "0",
    "--checkpoint_every", "10",
    "--early_stop_on", _early_stop,
    "--dropout", "0.1",
    "--hop_csv", str(hop_csv),
    "--attn_reg_territory_bias",
    "--attn_reg_territory_beta", str(ATTN_REG_TERRITORY_BETA),
]

if PHYSICS_WEIGHT > 0:
    _pf_flags = [
        "--pf_data_root", str(PF_DATA_ROOT),
        "--loss_power_balance_weight", str(PHYSICS_WEIGHT),
        "--pf_sparse_y", "1",
        "--pf_huber_delta_kw", "10",
        "--pf_detach_controls",
    ]
    _bal = Path(PF_BALANCE_NODE_LIST_CSV)
    if not _bal.is_absolute():
        _bal = (REPO / _bal).resolve()
    if not _bal.is_file():
        raise FileNotFoundError(f"PF_BALANCE_NODE_LIST_CSV not found: {_bal}")
    _pf_flags.extend(["--pf_balance_node_list_csv", str(_bal)])
    cmd.extend(_pf_flags)

_mode = "physics-informed" if PHYSICS_WEIGHT > 0 else "baseline"
print(f"PHYSICS_WEIGHT={PHYSICS_WEIGHT} ({_mode})")
print(f"early_stop_on={_early_stop}")
print("\nRunning:\n ", " ".join(cmd), "\n", flush=True)

with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nTraining completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Regulator classes:", (out_dir / "reg_class_tables.json").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())
_run_name = out_dir.name
print("\n=== Section 8 prerequisite ===")
print(f"Baseline run folder:  {out_dir.resolve()}")
print(f"Section 8 expects:    {MYDRIVE_DATA / 'runs' / _run_name}")
print("Checkpoint for init:  da_gps_multitask_best.pt (or training_last.pt)")
if _on_colab() and not str(out_dir.resolve()).startswith(str(MYDRIVE_DATA.resolve())):
    print(
        "WARNING: run dir is NOT under MyDrive/datasets_gnn2 — "
        "copy to Drive before disconnecting the VM or Section 8 will fail."
    )
else:
    print("Checkpoints are on Drive; safe to run Section 8 after this session ends.")
_write_smoke_compare_manifest(
    runs_parent,
    out_dir=out_dir,
    physics_weight=PHYSICS_WEIGHT,
    chunk_glob=CHUNK_GLOB,
    n_chunks=len(chunks),
    epochs=EPOCHS,
    seed=SEED,
)
